# Lab 04 (cluster): Exact evaluation vs. AI as a judge

**Chapter 3, "Evaluation Methodology"** (sections *Exact Evaluation*, *Introduction to Embedding*, *AI as a Judge*, *Ranking Models with Comparative Evaluation*)
**Environment:** your Docker JupyterLab on the KU VPN (course cluster) | **Time:** about 2.5-3 hours (the Colab notebook is the other part of this lab)

You are an AI engineer at Larkspur Ridge IT. Before you can improve the help-desk assistant you need a way to **score its answers**. In this notebook you score answers four ways (exact match, token F1, embedding similarity, and an LLM judge), then put the judge itself on trial: does a model prefer its own answers, and does it prefer whichever answer it reads first?

**Before you start**
- KU VPN on, Docker lab running, and `00_setup_check.ipynb` passed.
- Read Chapter 3 first. The analysis questions ask you to cite it.
- This notebook makes about **190 requests** and takes about 10-15 minutes of model time in total. Results are cached in `lab04_cache.json`, so re-running a cell does not re-send the same request. If you **change** a prompt (for example your rubric), the new prompt is a new request, so you get fresh answers; each full rubric change costs about 80 judge requests.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Evaluation metric** | A way to turn "is this answer good?" into a number. | You can't improve an assistant you can't score. |
| **Reference answer, gold set** | A correct answer written by a person. The **gold set** is a fixed list of questions with reference answers (the course's is in `eval/gold_qa.jsonl`). | Every metric in this lab compares the model's answer to a reference. |
| **Exact match** | 1 if the answer is the same text as the reference (after cleaning up case and punctuation), else 0. | Strict and simple, but it fails on correct answers worded differently. |
| **Token F1 (precision, recall)** | Partial credit for word overlap. Precision: how much of the answer is in the reference. Recall: how much of the reference the answer covered. F1 combines the two. | A **lexical** (word-based) metric: it sees words, not meaning. |
| **Embedding, vector** | An **embedding** model turns a text into a **vector** (a long list of numbers) so that texts with similar meaning get similar vectors. | It lets you compare meaning instead of words. |
| **Cosine similarity** | A score for how closely two vectors point the same way: about 1 = very similar meaning, 0 = unrelated. | The **semantic** (meaning-based) metric in this lab. |
| **AI as a judge (LLM judge)** | Asking a language model to grade an answer. | Grades answers exact metrics can't, but it has its own biases. |
| **Rubric** | The judge's instructions: what to check and what each score means. | The judge is only as good as its rubric. |
| **JSON, schema** | **JSON** is a simple text format for data, like `{"score": 4}`. A **schema** says which fields it must have. | Asking for JSON makes the judge's reply easy for code to read. |
| **Pointwise vs pairwise** | Pointwise: the judge scores one answer at a time. Pairwise: the judge sees two answers and picks the better one. | You test the judge both ways in B3. |
| **Self-bias, position bias** | Self-bias: a judge favors answers from its own model. Position bias: in a pairwise comparison, it favors whichever answer comes first (or second). | B3 tests both. |
| **Threshold** | A cutoff that turns a score into pass/fail, such as "F1 of 0.5 or more counts as correct". | Needed to compare metrics with your own yes/no labels. |
| **Percent agreement, Cohen's kappa** | Agreement: how often a metric and you gave the same label. Kappa: agreement after removing what you'd get by luck (0 = chance, 1 = perfect). | Shows which metric you could trust in place of a person. |
| **Cache** | A saved copy of each reply, so asking again doesn't send a new request. | Re-running cells is free and stays under your rate limit. |

## Setup

In [ ]:
import json, re, time, math, random, hashlib
from collections import Counter

import numpy as np
import pandas as pd

from itec632 import get_client, MODELS, NO_THINK, strip_think
from itec632.grading import LabChecker
from itec632.corpus import corpus_dir

client = get_client()
ck = LabChecker("lab04", part="cluster")
print("Models on the gateway:", [m.id for m in client.models.list().data])

The next cell defines a small `chat()` helper you will use everywhere. It is boilerplate, so you don't need to change it, but read the docstring:
- it waits about 1.5 seconds between calls so you stay under the 60 requests/minute limit,
- it saves every reply in a file (a **cache**) under a key plus a fingerprint of the exact prompt, so re-running is free, and editing a prompt (your rubric, `build_messages()`) automatically makes a new request instead of reusing an old reply,
- it asks for **structured JSON** when you pass a `schema`, and quietly falls back to plain text if the server refuses. (JSON is a simple text format for data, like `{"score": 4}`; a **schema** says which fields the reply must have.)

The same cell defines `embed()`, which sends texts to the course `embed` model and gets back one vector per text (A3 explains vectors).

In [ ]:
CACHE_FILE = "lab04_cache.json"
try:
    CACHE = json.load(open(CACHE_FILE))
except Exception:
    CACHE = {}
REQUESTS = {"chat": 0, "embed": 0}
MIN_GAP = 1.5            # seconds between calls (about 40 requests/minute)
_last_call = [0.0]


def chat(model, messages, max_tokens=300, temperature=0.0, schema=None, cache_key=None):
    """Send one chat request and return the reply text ("" if every attempt failed).

    model: a gateway alias such as MODELS["small"]; messages: OpenAI-style list of dicts;
    schema: optional JSON schema for structured output; cache_key: a name for this request.
    The cache uses the name PLUS a fingerprint of the messages, so the same name with the same
    prompt -> cached reply (no new request), but a changed prompt -> a new request.
    """
    if cache_key:
        fingerprint = hashlib.sha256(json.dumps([model, messages]).encode()).hexdigest()[:10]
        cache_key = f"{cache_key}|{fingerprint}"
    if cache_key and cache_key in CACHE:
        return CACHE[cache_key]
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens,
                  temperature=temperature, extra_body=NO_THINK)
    if schema is not None:
        kwargs["response_format"] = {"type": "json_schema",
                                     "json_schema": {"name": "result", "schema": schema}}
    text = ""
    for attempt in range(3):
        wait = MIN_GAP - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call[0] = time.time()
            REQUESTS["chat"] += 1
            r = client.chat.completions.create(**kwargs)
            text = strip_think(r.choices[0].message.content or "")
            break
        except Exception as e:
            msg = str(e)
            if "response_format" in kwargs and ("response_format" in msg or "schema" in msg or "400" in msg):
                kwargs.pop("response_format")       # structured output not supported: use plain text
                continue
            print(f"   (attempt {attempt + 1} failed: {msg[:100]})")
            time.sleep(5 * (attempt + 1))
    if cache_key and text:
        CACHE[cache_key] = text
        json.dump(CACHE, open(CACHE_FILE, "w"))
    return text


def embed(texts):
    """Embed a list of strings with the course embedding model. Returns an (n, d) numpy array."""
    out = []
    for i in range(0, len(texts), 32):
        REQUESTS["embed"] += 1
        r = client.embeddings.create(model=MODELS["embed"], input=texts[i:i + 32])
        out.extend(d.embedding for d in r.data)
    return np.array(out, dtype=float)


print(chat(MODELS["small"], [{"role": "user", "content": "Say hello in five words."}], max_tokens=20))

### The gold evaluation set

`corpus/larkspur-corpus/eval/gold_qa.jsonl` holds questions about Larkspur with human-written reference answers (see the `eval/README.md`). `gold_contexts.jsonl` gives the document sections that contain each answer, so you can test **the model's answering**, not retrieval (finding the right document is Lab 08's job). Look at a few rows.

**Reading the table:** one row per question. `qid` is the question's ID, `category` its topic, `difficulty` a rough rating, and `reference_answer` the correct answer written by a person. `trap` is `True` when the corpus also has an outdated document that gives a different answer (you'll meet those in Lab 05).

In [ ]:
EVAL = corpus_dir() / "eval"
GOLD = [json.loads(line) for line in open(EVAL / "gold_qa.jsonl")]
CTX = {}
for line in open(EVAL / "gold_contexts.jsonl"):
    rec = json.loads(line)
    CTX[rec["qid"]] = rec["passages"]
BY_ID = {q["qid"]: q for q in GOLD}


def is_unanswerable(q):
    return q["reference_answer"].startswith("Not in the documents")


def format_context(qid):
    """Join the gold passages for a question into one string, with doc id, version and date."""
    parts = [f"[{p['doc_id']} v{p['version']}, effective {p['effective_date']}] {p['title']} - {p['section']}\n{p['text']}"
             for p in CTX.get(qid, [])]
    return "\n\n---\n\n".join(parts) if parts else "(no documents)"


print(len(GOLD), "questions")
pd.DataFrame(GOLD)[["qid", "category", "trap", "difficulty", "question", "reference_answer"]].head(8)

# Part A - Guided practice

In Part A the metric code is **given**. An IT professional rarely writes a metric from scratch; the job is to know what each number means, predict where it goes wrong, and decide which one to trust. So in each section you run the metric, predict before you look, and make a call.

## A1. Exact match

Chapter 3 calls evaluation **exact** when it produces a judgment without ambiguity. The simplest version compares the model's answer to the reference character by character. That only works after you **normalize** both strings, and even then it fails whenever the model words a correct answer differently. Let's see how badly.

`normalize()` (given) lowercases the text, removes punctuation (but keeps `$` and the `.` or `:` inside a number, so `$247.50` and `7:00` survive), drops the words *a*, *an* and *the*, and collapses extra spaces. `exact_match()` returns 1 if the two normalized strings are identical, else 0.

In [ ]:
def normalize(text):
    """Lowercase, strip punctuation (keeping $ and decimal points/colons inside numbers),
    drop the articles a/an/the, and collapse whitespace."""
    t = (text or "").lower()
    t = re.sub(r"(?<!\d)[.:]|[.:](?!\d)", " ", t)   # . and : that are not inside a number
    t = re.sub(r"[^\w\s$.:]", " ", t)               # other punctuation -> space
    t = " ".join(w for w in t.split() if w not in {"a", "an", "the"})
    return t


def exact_match(pred, ref):
    return int(normalize(pred) == normalize(ref))


print(normalize("The answer: $247.50 per hour!"))

Below are four answers to *"What is the P2 response target on the Essentials plan?"* The reference is `2 business hours.` and `CORRECT` records what a person would say about each answer.

**TODO (before you run the table):** predict what `exact_match()` returns for each answer (1 or 0). Your prediction is not graded on being right.

In [ ]:
EM_REF = "2 business hours."
EM_PAIRS = {
    "a": "2 Business Hours",
    "b": "The Essentials P2 response target is 2 business hours.",
    "c": "3 business hours.",
    "d": "Two business hours.",
}
CORRECT = {"a": True, "b": True, "c": False, "d": True}   # a person's verdict

In [ ]:
EM_PREDICT = {"a": None, "b": None, "c": None, "d": None}   # TODO: your predicted exact_match result (1 or 0) for each answer

In [ ]:
em_table = pd.DataFrame([{"pair": k, "answer": a, "correct?": CORRECT[k], "your prediction": EM_PREDICT.get(k),
                          "exact_match": exact_match(a, EM_REF)} for k, a in EM_PAIRS.items()]).set_index("pair")
display(em_table)

**Reading the table:** one row per answer. `correct?` is the person's verdict, `exact_match` is the metric's. A **miss** (false negative) is a correct answer that the metric scored 0.

**TODO:** list the letters of the answers exact match missed.

In [ ]:
EM_MISSED = []    # TODO: letters of the answers that are correct but got exact_match 0, e.g. ["a"]

In [ ]:
ck.check("A1_predictions", all(EM_PREDICT.get(k) in (0, 1) for k in EM_PAIRS), points=1,
         hint="Set every value in EM_PREDICT to 1 or 0 (your guess, before looking at the table).")
truly_missed = {k for k, a in EM_PAIRS.items() if CORRECT[k] and exact_match(a, EM_REF) == 0}
ck.check("A1_misses_found", set(EM_MISSED) == truly_missed, points=2,
         hint="Look for rows where correct? is True but exact_match is 0. List exactly those letters, and no others.")
ck.record("A1_predictions_right", sum(EM_PREDICT.get(k) == exact_match(a, EM_REF) for k, a in EM_PAIRS.items()))

## A2. Lexical similarity: token F1

Lexical similarity gives partial credit for overlapping tokens (the chapter discusses n-gram overlap metrics such as BLEU and ROUGE; token F1 is the simplest member of that family and is the standard metric for short-answer QA). Here a "token" is simply a word after `normalize()`. `token_f1()` (given) counts the words the answer and the reference share, then combines two ratios:

- **precision**: how much of what the model said is in the reference (shared words / words in the answer),
- **recall**: how much of the reference the model covered (shared words / words in the reference),
- **F1**: a balance of the two that is high only when both are high (0 = no shared words, 1 = the same words).

In [ ]:
def token_f1(pred, ref):
    """Word-overlap F1 between an answer and a reference (0 to 1)."""
    p, r = normalize(pred).split(), normalize(ref).split()
    overlap = sum((Counter(p) & Counter(r)).values())     # shared words, counting repeats
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(r)
    return 2 * precision * recall / (precision + recall)


print(token_f1("2 business hours", "2 business hours"))
print(token_f1("3 business hours", "2 business hours"), "<- a WRONG answer still scores this")
print(token_f1("Within two hours during the business day", "2 business hours"), "<- a right answer, reworded")

**TODO: try to fool the metric.** Testing a metric the way a tester tests software is a real evaluation skill: find inputs where it gives the wrong verdict. The reference below is the after-hours rate. Suppose Larkspur counted any answer with token F1 of 0.5 or more as correct. Write:
- `MY_WRONG_HIGH`: an answer that is **wrong** for a Larkspur employee but still scores **0.5 or more**;
- `MY_RIGHT_LOW`: an answer that is **correct** but scores **below 0.5**.

Use your own wording, not the examples above. Run the cell to see your scores, and adjust until both land on the wrong side of 0.5.

In [ ]:
F1_REF = "$247.50 per hour, with a 1-hour minimum."
MY_WRONG_HIGH = ""    # TODO: a WRONG answer that still gets token F1 >= 0.5 against F1_REF
MY_RIGHT_LOW = ""     # TODO: a CORRECT answer that gets token F1 < 0.5
for name, text in [("MY_WRONG_HIGH", MY_WRONG_HIGH), ("MY_RIGHT_LOW", MY_RIGHT_LOW)]:
    print(f"{name:14} token_f1 = {token_f1(text, F1_REF):.2f}   {text!r}")

In [ ]:
f1_wrong, f1_right = token_f1(MY_WRONG_HIGH, F1_REF), token_f1(MY_RIGHT_LOW, F1_REF)
ck.check("A2_fooled_token_f1",
         bool(MY_WRONG_HIGH.strip()) and bool(MY_RIGHT_LOW.strip()) and f1_wrong >= 0.5 and f1_right < 0.5
         and normalize(MY_WRONG_HIGH) != normalize(F1_REF),
         points=2, hint="MY_WRONG_HIGH needs token F1 of 0.5 or more (keep most words, change the key fact); "
                        "MY_RIGHT_LOW needs less than 0.5 (same fact, different words). The printed scores show where you are.")
ck.record("A2_f1_wrong_high", f1_wrong)
ck.record("A2_f1_right_low", f1_right)

## A3. Semantic similarity with embeddings

Embedding similarity compares meaning instead of surface words (see *Introduction to Embedding*). The course `embed` model turns text into a **vector** (a long list of numbers) so that texts with similar meaning get similar vectors. **Cosine similarity** (given in `cosine()`) measures how closely two vectors point the same way: close to 1 for similar meaning, around 0 for unrelated text.

Below are three candidate answers to the after-hours rate question: a correct answer in different words, an answer with the **wrong** price, and an unrelated sentence.

**TODO (before you run the comparison):** predict which candidate will have the **highest** cosine similarity to the reference.

In [ ]:
def cosine(u, v):
    """Cosine similarity of two vectors (1 = same direction, 0 = unrelated, -1 = opposite)."""
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(u @ v / (nu * nv))


ref = "$247.50 per hour, with a 1-hour minimum."
candidates = {
    "correct, reworded": "It is billed at 247.50 dollars an hour and there is a one hour minimum.",
    "WRONG number":      "$210 per hour, with a 1-hour minimum.",
    "unrelated":         "Priya Raman is the Service Desk Manager.",
}

In [ ]:
PREDICT_CLOSEST = ""    # TODO: "correct, reworded", "WRONG number", or "unrelated"

In [ ]:
vecs = embed([ref] + list(candidates.values()))
sims = {name: cosine(vecs[0], vecs[i + 1]) for i, name in enumerate(candidates)}
for name, s in sims.items():
    print(f"{name:18} cosine={s:.3f}   token_f1={token_f1(candidates[name], ref):.2f}")
closest = max(sims, key=sims.get)
print(f"\nHighest cosine: {closest!r}   (you predicted {PREDICT_CLOSEST or 'nothing'!r})")
ck.record("A3_sim_correct", sims["correct, reworded"])
ck.record("A3_sim_wrong_number", sims["WRONG number"])
ck.record("A3_prediction_right", PREDICT_CLOSEST == closest)

**Reading the output:** one line per candidate answer, compared with the reference `$247.50 per hour, with a 1-hour minimum.` `cosine` is the embedding similarity and `token_f1` the word-overlap score from A2. Compare the "correct, reworded" and "WRONG number" lines: which metric ranks them the right way round, and how far apart are they?

**TODO: make a call.** Many Larkspur questions have a **price, time, or phone number** as the answer, where one wrong digit makes the answer wrong. Which metric would you trust to score those automatically: `"exact_match"`, `"token_f1"`, `"embedding"`, or `"none"` (none of these three is good enough on its own)? Give your reason in one or two sentences, using what you saw in A1-A3.

In [ ]:
METRIC_FOR_NUMBERS = {
    "metric": "",   # TODO: "exact_match", "token_f1", "embedding", or "none"
    "why": "",      # TODO: one or two sentences, based on your A1-A3 results
}

In [ ]:
ck.check("A3_prediction", PREDICT_CLOSEST in candidates, points=1,
         hint='Set PREDICT_CLOSEST to one of the three candidate names, spelled exactly: "correct, reworded", "WRONG number", or "unrelated".')
ck.check("A3_metric_decision", METRIC_FOR_NUMBERS.get("metric") in ("exact_match", "token_f1", "embedding", "none")
         and len(str(METRIC_FOR_NUMBERS.get("why", "")).split()) >= 8, points=1,
         hint='Set "metric" to one of the four options and write at least one full sentence (8+ words) in "why".')

## A4. AI as a judge, with a rubric

An LLM judge can grade answers that exact metrics can't, but the chapter warns that its scores are only as good as the **prompt and the scoring rubric**, and that a judge's scores are not comparable once you change either (*Limitations of AI as a Judge: criteria ambiguity, inconsistency*).

A good judge prompt (chapter, *How to Use AI as a Judge*) states the task, the criteria, the scoring scale with what each score means, and the output format.

**TODO:** finish the rubric. Replace the three `<describe ...>` placeholders with your own definitions of a 5, a 3, and a 1 (you may add 4 and 2). Keep the `{question}`, `{reference}`, and `{answer}` placeholders. Think about Larkspur: is an answer with the right fact but an extra wrong detail a 3 or a 1? What about "Not in the documents" when the reference has an answer?

In [ ]:
RUBRIC = """You are grading an answer from Larkspur Ridge IT's help-desk assistant.
Compare the ASSISTANT ANSWER to the REFERENCE ANSWER, which is correct and current.
Judge factual correctness only: does the answer state the same key facts (numbers, names, times, steps)?
Ignore wording, length, and politeness.

Score from 1 to 5:
5 = <describe a 5>
3 = <describe a 3>
1 = <describe a 1>

QUESTION: {question}
REFERENCE ANSWER: {reference}
ASSISTANT ANSWER: {answer}

Reply with JSON only: {"score": <1-5>, "reason": "<one sentence>"}"""
# TODO: replace the three <describe ...> placeholders above with your own score definitions.

Judges don't always follow the output format, especially small models. `parse_score()` below tries JSON first, then patterns like `Score: 4` or `4/5`, and returns `None` when it can't find a valid score. **Always count how many judgments failed to parse**; silently dropping them biases your averages.

In [ ]:
JUDGE_SCHEMA = {"type": "object",
                "properties": {"score": {"type": "integer", "minimum": 1, "maximum": 5},
                               "reason": {"type": "string"}},
                "required": ["score", "reason"]}


def parse_score(text, lo=1, hi=5):
    """Extract an integer score from a judge reply, or None if there isn't a valid one."""
    if not text:
        return None
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            v = int(round(float(json.loads(m.group(0)).get("score"))))
            if lo <= v <= hi:
                return v
        except Exception:
            pass
    m = re.search(r"score\W{0,5}([1-9])", text, re.I) or re.search(r"\b([1-9])\s*(?:/|out of)\s*5\b", text)
    if m and lo <= int(m.group(1)) <= hi:
        return int(m.group(1))
    return None


def judge_answer(question, reference, answer, judge_model, key):
    """Score one answer with an LLM judge. Returns (score or None, raw reply)."""
    prompt = (RUBRIC.replace("{question}", question).replace("{reference}", reference)
                    .replace("{answer}", answer or "(empty answer)"))
    raw = chat(judge_model, [{"role": "user", "content": prompt}], max_tokens=200,
               schema=JUDGE_SCHEMA, cache_key=f"judge|{judge_model}|{key}")
    return parse_score(raw), raw


for t in ['{"score": 4, "reason": "ok"}', "Score: 2 - wrong number", "I'd give it 5/5", "great answer!", ""]:
    print(repr(t), "->", parse_score(t))

In [ ]:
q = "What is the hourly rate for out-of-scope work done after hours?"
ref = "$247.50 per hour (1.5x), with a 1-hour minimum."
tests = {"correct": "It's $247.50 an hour with a one-hour minimum.",
         "wrong_number": "After-hours work is billed at $210 per hour.",
         "verbose_correct": "Great question! Larkspur bills after-hours, weekend, and holiday work at $247.50 per hour, "
                            "which is 1.5 times the normal rate, and there is a 1-hour minimum. Let me know if you need anything else!"}
demo_scores = {}
for name, ans in tests.items():
    s, raw = judge_answer(q, ref, ans, MODELS["main"], key=f"demo|{name}")
    demo_scores[name] = s
    print(f"{name:16} score={s}   raw={raw[:90]!r}")

**Reading the output:** one line per test answer. `score` is the 1-5 score that `parse_score()` read from the judge's reply (`None` if it couldn't find one), and `raw` is the start of the reply itself. Compare `correct` with `verbose_correct`: same facts, but one is padded with chatter. If your judge scores them differently, that is **verbosity bias** (a judge favoring longer answers), which the chapter lists among judge biases. Keep the number in mind for Q2.

If you change the rubric later, just re-run these cells: the new rubric is a new prompt, so the judge is asked again.

In [ ]:
score_lines = {n: re.search(rf"^\s*{n}\s*=\s*(.+)$", RUBRIC, re.M) for n in "531"}
rubric_done = ("<describe" not in RUBRIC and all(p in RUBRIC for p in ["{question}", "{reference}", "{answer}"])
               and all(m is not None and len(m.group(1).split()) >= 5 for m in score_lines.values()))
ck.check("A4_rubric_written", rubric_done, points=2,
         hint="Replace every <describe ...> placeholder with a real definition (at least 5 words for each of the 5, 3 and 1 lines) "
              "and keep {question}, {reference}, {answer}.")
parsed = [s for s in demo_scores.values() if s is not None]
ck.check("A4_judge_parsed", len(parsed) >= 2 and (demo_scores["correct"] or 0) >= (demo_scores["wrong_number"] or 0),
         points=2, hint="At least 2 of 3 judgments should parse, and the correct answer should not score below the wrong one. "
                         "Print the raw replies; is your rubric asking for JSON?")

# Part B - Applied lab: scoring the help-desk assistant

## B1. Pick the questions and get answers from `qwen-small`

You'll score `qwen-small` (the cheap, fast model) on 25 questions: 22 answerable questions plus 3 that the documents **can't** answer. Each question comes with its gold context. (Trap questions, where the corpus has an outdated document too, are saved for Lab 5.)

**TODO:** complete `build_messages()` so the model gets the system prompt, then the context and the question.

In [ ]:
answerable = [q for q in GOLD if q["category"] != "data" and not q["trap"] and not is_unanswerable(q)]
unanswerable = [q for q in GOLD if is_unanswerable(q)]
rng = random.Random(4)
SUBSET = rng.sample(answerable, 22) + rng.sample(unanswerable, 3)
print(len(SUBSET), "questions;", Counter(q["category"] for q in SUBSET))

ANSWER_SYSTEM = ("You are the Larkspur Ridge IT help-desk assistant. Answer using ONLY the documents provided. "
                 "Answer in one or two sentences with the key fact first. "
                 "If the documents do not contain the answer, reply exactly: Not in the documents.")

In [ ]:
def build_messages(q):
    """Messages for one gold question: system prompt, then documents + question."""
    # TODO: build a user message containing "Documents:", format_context(q["qid"]), and "Question: " + q["question"]
    # TODO: return [{"role": "system", "content": ANSWER_SYSTEM}, {"role": "user", "content": <your user message>}]
    return []

In [ ]:
ANSWERS = {}
if not build_messages(SUBSET[0]):
    print("Complete build_messages() above first.")
else:
    t0 = time.time()
    for i, q in enumerate(SUBSET, 1):
        ANSWERS[q["qid"]] = chat(MODELS["small"], build_messages(q), max_tokens=150, cache_key=f"ans|small|{q['qid']}")
        if i % 5 == 0:
            print(f"{i}/{len(SUBSET)} answered ({time.time() - t0:.0f}s)")
    for q in SUBSET[:3]:
        print(f"\n{q['qid']} {q['question']}\n  model: {ANSWERS[q['qid']]}\n  ref:   {q['reference_answer']}")

In [ ]:
ck.check("B1_answers", len([a for a in ANSWERS.values() if a]) >= 23, points=3,
         hint="You should have about 25 non-empty answers. Did build_messages() return two messages? Any errors printed?")

## B2. Score every answer four ways

Now compute exact match, token F1, embedding similarity, and the `qwen-moe` judge score for each answer (25 judge requests). Then **choose pass/fail thresholds** for F1 and embedding similarity (a **threshold** is a cutoff: for example, F1 of 0.5 or more counts as correct): you'll need them to compare against human labels in B4. Look at the table before choosing; there is no single right value.

**How to choose:** find two answers in the table that you would call correct and two you would call wrong. Pick an F1 cutoff that sits between them (most correct ones above, most wrong ones below), and do the same for `emb`. In B4 you'll see how well your cutoffs agreed with your own labels, so a rough choice is fine.

In [ ]:
rows = []
if ANSWERS:
    qs = [q for q in SUBSET if q["qid"] in ANSWERS]
    ref_vecs = embed([q["reference_answer"] for q in qs])
    ans_vecs = embed([ANSWERS[q["qid"]] or "(empty)" for q in qs])
    for q, rv, av in zip(qs, ref_vecs, ans_vecs):
        a = ANSWERS[q["qid"]]
        js, _ = judge_answer(q["question"], q["reference_answer"], a, MODELS["main"], key=f"small|{q['qid']}")
        rows.append({"qid": q["qid"], "category": q["category"], "unanswerable": is_unanswerable(q),
                     "em": exact_match(a, q["reference_answer"]), "f1": token_f1(a, q["reference_answer"]),
                     "emb": cosine(rv, av), "judge": js,
                     "answer": a, "reference": q["reference_answer"]})
scores = pd.DataFrame(rows)
if len(scores):
    scores["judge"] = pd.to_numeric(scores["judge"], errors="coerce")   # None -> NaN so comparisons work
    print("Judge parse failures:", int(scores["judge"].isna().sum()), "of", len(scores))
    display(scores[["qid", "category", "em", "f1", "emb", "judge", "answer", "reference"]].round(2))

**Reading the table:** one row per question. `em` is exact match (1 or 0), `f1` is token F1 (0 to 1), `emb` is the embedding cosine similarity between answer and reference (about 0 to 1), and `judge` is the `qwen-moe` judge's 1-5 score (`NaN` means its reply couldn't be parsed). `answer` is what `qwen-small` said and `reference` is the correct answer. Scroll right to read them.

In [ ]:
# TODO: after looking at the table, choose thresholds (numbers between 0 and 1)
F1_PASS = None    # token F1 at or above this counts as "correct"
EMB_PASS = None   # embedding cosine at or above this counts as "correct"

In [ ]:
def mean_or_none(series):
    s = pd.Series(series).dropna()
    return float(s.mean()) if len(s) else None


if len(scores):
    summary = {"exact_match": mean_or_none(scores["em"]), "token_f1": mean_or_none(scores["f1"]),
               "embedding_sim": mean_or_none(scores["emb"]), "judge_1to5": mean_or_none(scores["judge"])}
    print({k: (round(v, 3) if v is not None else None) for k, v in summary.items()})
    for k, v in summary.items():
        ck.record(f"B2_{k}", v)
    # Where do the metrics disagree? Judge says good (>=4) but exact match says 0:
    display(scores[(scores["judge"] >= 4) & (scores["em"] == 0)][["qid", "f1", "emb", "judge", "answer", "reference"]].head(5))

**Reading the output:** the printed line is each metric's average over all answers (exact match, F1, and embedding on a 0-1 scale, the judge on 1-5). The table lists up to five answers the judge rated 4 or 5 although exact match gave them 0: correct answers that exact match missed, or judge mistakes. It may be empty.

In [ ]:
ck.check("B2_scores_table", len(scores) >= 23 and scores[["em", "f1", "emb"]].notna().all().all(), points=2,
         hint="The scores table needs a row per answer with em, f1, and emb filled in. Did B1 finish?")
ck.check("B2_thresholds_chosen", isinstance(F1_PASS, (int, float)) and isinstance(EMB_PASS, (int, float))
         and 0 < F1_PASS < 1 and 0 < EMB_PASS < 1, points=1, hint="Set F1_PASS and EMB_PASS to numbers between 0 and 1.")

## B3. Is the judge biased?

The chapter lists judge biases, among them **self-bias** (a model favors its own responses over responses generated by other models; here each judge also wrote one of the two answer sets) and **position bias** (in a pairwise comparison, a judge favors the answer in one position). You'll test both with two judges from different families, `qwen-moe` and `gemma-moe`, on 12 questions.

**Why this matters at Larkspur:** next week (Lab 05) you choose a model for the help-desk assistant, partly by having an LLM judge score the candidates. If the judge quietly prefers answers from its own model family, the "winner" could be the judge's sibling rather than the best model.

**Step 1.** Both models answer the 12 questions (24 requests).
**Step 2. Pointwise:** each judge scores each model's answer with your rubric (48 requests). If there is no self-bias, both judges should see the *same* gap between the two models.
**Step 3. Pairwise:** each judge sees both answers side by side, then again with the order swapped (48 requests).

In [ ]:
rng2 = random.Random(11)
BIAS_Q = rng2.sample(answerable, 12)
AUTHORS = {"qwen": MODELS["main"], "gemma": MODELS["gemma"]}
JUDGES = {"qwen": MODELS["main"], "gemma": MODELS["gemma"]}

BIAS_ANS = {a: {} for a in AUTHORS}
if build_messages(BIAS_Q[0]):
    for q in BIAS_Q:
        for a, model in AUTHORS.items():
            BIAS_ANS[a][q["qid"]] = chat(model, build_messages(q), max_tokens=150, cache_key=f"ans|{a}|{q['qid']}")
    print("answers:", {a: len(v) for a, v in BIAS_ANS.items()})

# Pointwise: POINT[judge][author] = list of scores
POINT = {j: {a: [] for a in AUTHORS} for j in JUDGES}
for q in BIAS_Q:
    for j, jm in JUDGES.items():
        for a in AUTHORS:
            if q["qid"] in BIAS_ANS[a]:
                s, _ = judge_answer(q["question"], q["reference_answer"], BIAS_ANS[a][q["qid"]], jm, key=f"bias|{a}|{q['qid']}")
                POINT[j][a].append(s)
point_table = pd.DataFrame({f"judge={j}": {f"author={a}": mean_or_none(POINT[j][a]) for a in AUTHORS} for j in JUDGES})
print("Mean pointwise score (rows = who wrote the answer, columns = who judged it):")
display(point_table.round(2))
print("Parse failures:", {j: sum(s is None for a in AUTHORS for s in POINT[j][a]) for j in JUDGES})

### The self-preference index (given)

The next cell turns `point_table` into **one self-preference index for the pair of judges**.

The idea: each judge sees a *gap* between qwen's answers and gemma's answers. If qwen's answers really are better, **both** judges should see about the same gap. So the index subtracts one judge's gap from the other's and halves it: the real quality difference cancels out, and what's left is how much the judges lean toward their own family.

A worked example with made-up numbers: the qwen judge gives qwen answers 4.5 and gemma answers 4.0 (gap +0.5). The gemma judge gives qwen answers 4.2 and gemma answers 4.1 (gap +0.1). Both agree qwen is a bit better, but the qwen judge sees a bigger gap. Index = (0.5 - 0.1) / 2 = **+0.2**: a mild lean toward their own family.

How big is big? Scores are 1-5 averages over 12 questions, so one answer moving by one point shifts an average by about 0.08. An index within about +/-0.2 could easily be chance; 0.5 or more is a clear lean. Positive means the judges each favor their own family; negative means they each favor the other. Between 0.2 and 0.5 (either sign) is a judgment call.

In [ ]:
def cell(author, judge):
    """One number from point_table: the mean score `judge` gave to answers written by `author`."""
    v = point_table.loc[f"author={author}", f"judge={judge}"]
    return None if v is None or pd.isna(v) else float(v)


vals = [cell(a, j) for a in AUTHORS for j in JUDGES]
if any(v is None for v in vals):
    self_pref = None
else:
    gap_seen_by_qwen = cell("qwen", "qwen") - cell("gemma", "qwen")
    gap_seen_by_gemma = cell("qwen", "gemma") - cell("gemma", "gemma")
    self_pref = (gap_seen_by_qwen - gap_seen_by_gemma) / 2
    print(f"gap seen by the qwen judge: {gap_seen_by_qwen:+.2f}   gap seen by the gemma judge: {gap_seen_by_gemma:+.2f}")
print("self-preference index:", None if self_pref is None else round(self_pref, 3))

**TODO:** read your index and give the verdict a careful analyst would report, using the size guidance above: `"own family"`, `"other family"`, or `"no clear lean"`.

In [ ]:
SELF_BIAS_VERDICT = ""    # TODO: "own family", "other family", or "no clear lean"

In [ ]:
def self_verdict_ok(verdict, x):
    """Is the verdict defensible for index x under the size guidance?"""
    if x is None or verdict not in ("own family", "other family", "no clear lean"):
        return False
    if abs(x) <= 0.2:
        return verdict == "no clear lean"
    if abs(x) >= 0.5:
        return verdict == ("own family" if x > 0 else "other family")
    return verdict in ("no clear lean", "own family" if x > 0 else "other family")    # 0.2-0.5: your call


ck.check("B3_bias_answers", all(len(BIAS_ANS[a]) == 12 for a in AUTHORS), points=2,
         hint="Both qwen-moe and gemma-moe should have answered all 12 bias questions.")
ck.check("B3_pointwise_complete", point_table.notna().all().all(), points=2,
         hint="Every judge/author cell needs at least one parsed score. Check the parse failures printed above.")
ck.check("B3_self_bias_verdict", self_verdict_ok(SELF_BIAS_VERDICT, self_pref), points=1,
         hint="Compare the size of your printed index with the guidance above (about +/-0.2 could be chance; 0.5 or more is a clear lean; "
              "the sign says which way). If the index is None, fix the pointwise step first.")
ck.record("B3_self_pref_index", self_pref)
ck.record("B3_self_bias_verdict", SELF_BIAS_VERDICT)

### Pairwise comparison and position bias

The pairwise prompt asks which answer is more correct. We run every pair twice: qwen's answer in position A, then gemma's answer in position A. A judge without position bias picks the same *model* both times.

In [ ]:
PAIR_PROMPT = """You are comparing two answers from Larkspur Ridge IT's help-desk assistant.
The REFERENCE ANSWER is correct and current. Decide which answer is more factually correct and complete
compared with the reference. Ignore style and length. Answer "tie" only if they are equally correct.

QUESTION: {question}
REFERENCE ANSWER: {reference}
ANSWER A: {answer_a}
ANSWER B: {answer_b}

Reply with JSON only: {"winner": "A" or "B" or "tie", "reason": "<one sentence>"}"""
PAIR_SCHEMA = {"type": "object", "properties": {"winner": {"type": "string", "enum": ["A", "B", "tie"]},
                                               "reason": {"type": "string"}}, "required": ["winner", "reason"]}


def parse_winner(text):
    """Return "A", "B", "tie", or None from a pairwise judge reply."""
    if not text:
        return None
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            w = str(json.loads(m.group(0)).get("winner", "")).strip().upper()
            if w in ("A", "B"):
                return w
            if w == "TIE":
                return "tie"
        except Exception:
            pass
    m = re.search(r"winner\W{0,5}(A|B|tie)\b", text, re.I) or re.search(r"\banswer\s+(A|B)\s+is\s+(?:more|better)", text, re.I)
    if m:
        return "tie" if m.group(1).lower() == "tie" else m.group(1).upper()
    t = text.strip().strip(".").upper()
    return t if t in ("A", "B") else ("tie" if t == "TIE" else None)


PAIRS = []   # one row per (question, judge, order)
for q in BIAS_Q:
    if not all(q["qid"] in BIAS_ANS[a] for a in AUTHORS):
        continue
    for j, jm in JUDGES.items():
        for first in ("qwen", "gemma"):
            second = "gemma" if first == "qwen" else "qwen"
            prompt = (PAIR_PROMPT.replace("{question}", q["question"]).replace("{reference}", q["reference_answer"])
                      .replace("{answer_a}", BIAS_ANS[first][q["qid"]]).replace("{answer_b}", BIAS_ANS[second][q["qid"]]))
            raw = chat(jm, [{"role": "user", "content": prompt}], max_tokens=150, schema=PAIR_SCHEMA,
                       cache_key=f"pair|{j}|{first}|{q['qid']}")
            w = parse_winner(raw)
            winner_model = {"A": first, "B": second}.get(w, w)   # "qwen", "gemma", "tie", or None
            PAIRS.append({"qid": q["qid"], "judge": j, "first": first, "pos_winner": w, "winner_model": winner_model})
pairs = pd.DataFrame(PAIRS)
if len(pairs):
    print("Parse failures:", int(pairs["pos_winner"].isna().sum()), "of", len(pairs))
    display(pairs.groupby(["judge", "first"])["winner_model"].value_counts().unstack(fill_value=0))

**Reading the table:** each row is one judge with one order: `first` says whose answer was shown in position A. The columns count how many of the 12 questions each model's answer won (a `tie` column appears if there were ties). A judge without position bias gives similar counts in its two rows. If the winner mostly flips to whichever model was shown first, that's position bias.

The next cell shows one question in full, so you can see what the judges were actually comparing. It picks a question where a judge changed its mind when the order was swapped, if there is one.

In [ ]:
if len(pairs):
    flips = [qid for (qid, j), g in pairs.groupby(["qid", "judge"]) if g["winner_model"].nunique() > 1]
    show = flips[0] if flips else pairs["qid"].iloc[0]
    q = BY_ID[show]
    print(f"{show}: {q['question']}\nREFERENCE: {q['reference_answer']}\n")
    for a in AUTHORS:
        print(f"{a} answer: {BIAS_ANS[a][show]}\n")
    for _, r in pairs[pairs["qid"] == show].iterrows():
        print(f"judge={r['judge']:6} shown first={r['first']:6} -> picked position {r['pos_winner']} = {r['winner_model']}")
    print("\n(A changed pick between the two orders = position bias on this question.)" if flips
          else "\n(No judge changed its pick on any question: no position bias visible here.)")

### Position statistics (given)

`position_stats()` computes two numbers for each judge:
- `first_pos_rate`: among judgments that picked A or B (no ties, no parse failures), the fraction that picked **A**, the answer shown first. No position bias -> about 0.5.
- `consistency`: among questions where both orders parsed, the fraction where the judge picked the **same model** (`winner_model`) in both orders. A judge with no position bias is consistent every time.

In [ ]:
def position_stats(df, judge):
    """(first_pos_rate, consistency) for one judge; None where there is nothing to compute."""
    d = df[df["judge"] == judge]
    picked = d[d["pos_winner"].isin(["A", "B"])]
    first_pos_rate = float((picked["pos_winner"] == "A").mean()) if len(picked) else None

    same, total = 0, 0
    for qid, g in d.dropna(subset=["winner_model"]).groupby("qid"):
        if len(g) == 2:                       # both orders gave a parsed verdict
            total += 1
            same += int(g["winner_model"].iloc[0] == g["winner_model"].iloc[1])
    consistency = same / total if total else None
    return first_pos_rate, consistency


POS = {}
for j in JUDGES:
    POS[j] = position_stats(pairs, j) if len(pairs) else (None, None)
    rate, cons = POS[j]
    print(f"judge={j:6} first-position win rate={None if rate is None else round(rate, 2)}  "
          f"order consistency={None if cons is None else round(cons, 2)}")
    ck.record(f"B3_first_pos_rate_{j}", rate)
    ck.record(f"B3_consistency_{j}", cons)

**TODO:** for each judge, give your verdict on position bias: `"favors A"` (the first answer), `"favors B"` (the second), or `"no clear bias"`.

Guidance: each judge made about 24 pairwise picks. A first-position rate between 0.4 and 0.6 is within what chance easily produces, so call it `"no clear bias"`. A rate of 0.75 or more is a clear lean toward A, and 0.25 or less toward B. In between is your call; low order consistency is a reason to lean toward calling it a bias.

In [ ]:
POSITION_VERDICT = {"qwen": "", "gemma": ""}    # TODO: for each judge, "favors A", "favors B", or "no clear bias"

In [ ]:
def position_verdict_ok(verdict, rate):
    """Is the verdict defensible for this first-position rate under the guidance?"""
    if rate is None or verdict not in ("favors A", "favors B", "no clear bias"):
        return False
    if 0.4 <= rate <= 0.6:
        return verdict == "no clear bias"
    if rate >= 0.75:
        return verdict == "favors A"
    if rate <= 0.25:
        return verdict == "favors B"
    return verdict in ("no clear bias", "favors A" if rate > 0.5 else "favors B")    # in between: your call


ck.check("B3_position_verdicts", all(position_verdict_ok(POSITION_VERDICT.get(j), POS[j][0]) for j in JUDGES), points=2,
         hint="Compare each judge's printed first-position win rate with the guidance (0.4-0.6 no clear bias; 0.75+ favors A; "
              "0.25 or less favors B). If a rate is None, check the pairwise parse failures above.")
for j in JUDGES:
    ck.record(f"B3_position_verdict_{j}", POSITION_VERDICT.get(j))

## B4. Do the metrics agree with a human (you)?

An automatic metric is only useful if it agrees with what a careful person would say. Below are 10 of `qwen-small`'s answers with their references. **Read each one and label it yourself: 1 = correct (a Larkspur employee could act on it), 0 = not correct.** Do it by reading, not by asking an AI; you are the ground truth here.

In [ ]:
LABEL_IDS = [q["qid"] for q in SUBSET if not is_unanswerable(q)][:10]
for qid in LABEL_IDS:
    print(f"{qid}: {BY_ID[qid]['question']}\n   REFERENCE: {BY_ID[qid]['reference_answer']}\n   MODEL:     {ANSWERS.get(qid, '(no answer)')}\n")

In [ ]:
# TODO: replace each None with 1 (correct) or 0 (not correct), based on YOUR reading of the answers above.
HUMAN = {qid: None for qid in LABEL_IDS}
print(HUMAN)

A second judge (`gemma-moe`) also scores these 10 answers (10 requests), so you can compare judges. The cell then computes **percent agreement** and **Cohen's kappa** between your labels and each automatic method. Kappa corrects for agreement by chance: 0 means no better than chance, 1 means perfect.

In [ ]:
def cohen_kappa(a, b):
    """Cohen's kappa for two lists of 0/1 labels. Returns None if undefined."""
    pairs_ = [(x, y) for x, y in zip(a, b) if x is not None and y is not None]
    n = len(pairs_)
    if n == 0:
        return None
    po = sum(x == y for x, y in pairs_) / n
    pa, pb = sum(x for x, _ in pairs_) / n, sum(y for _, y in pairs_) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return None if pe == 1 else (po - pe) / (1 - pe)


labeled = [qid for qid in LABEL_IDS if HUMAN.get(qid) in (0, 1)]
gemma_scores = {}
for qid in labeled:
    q = BY_ID[qid]
    gemma_scores[qid], _ = judge_answer(q["question"], q["reference_answer"], ANSWERS.get(qid, ""), MODELS["gemma"], key=f"small|{qid}")

agree_rows = []
if labeled and len(scores):
    s = scores.set_index("qid")
    f1_cut = F1_PASS if isinstance(F1_PASS, (int, float)) else 0.5
    emb_cut = EMB_PASS if isinstance(EMB_PASS, (int, float)) else 0.8
    methods = {
        "exact_match": {qid: int(s.loc[qid, "em"]) for qid in labeled},
        f"token_f1>={f1_cut}": {qid: int(s.loc[qid, "f1"] >= f1_cut) for qid in labeled},
        f"embedding>={emb_cut}": {qid: int(s.loc[qid, "emb"] >= emb_cut) for qid in labeled},
        "judge_qwen>=4": {qid: (None if pd.isna(s.loc[qid, "judge"]) else int(s.loc[qid, "judge"] >= 4)) for qid in labeled},
        "judge_gemma>=4": {qid: (None if gemma_scores[qid] is None else int(gemma_scores[qid] >= 4)) for qid in labeled},
    }
    human = [HUMAN[qid] for qid in labeled]
    for name, m in methods.items():
        pred = [m[qid] for qid in labeled]
        valid = [(h, p) for h, p in zip(human, pred) if p is not None]
        agree = sum(h == p for h, p in valid) / len(valid) if valid else None
        agree_rows.append({"method": name, "n": len(valid), "agreement": agree, "kappa": cohen_kappa(human, pred)})
agreement = pd.DataFrame(agree_rows)
display(agreement.round(2) if len(agreement) else "Label the items above first.")

**Reading the table:** one row per automatic method. `n` is how many of your labels it could be compared on, `agreement` is the fraction where it gave the same 1/0 label as you, and `kappa` is Cohen's kappa (agreement beyond chance; `None` when it can't be computed, for example when every label is the same). If you labeled all 10 answers correct, kappa is `None` for every method: with no wrong answers there is nothing for a metric to catch. That is a real finding about small evaluation sets (say so in Q3) and not an error in your notebook.

In [ ]:
ck.check("B4_human_labels", len(labeled) >= 10, points=3,
         hint="Label all 10 items in HUMAN with 1 or 0.")
ck.check("B4_agreement_table", len(agreement) == 5 and agreement["agreement"].notna().sum() >= 4, points=2,
         hint="The agreement table should have 5 methods with agreement values. Did you label the items and run B2?")
for _, r in agreement.iterrows():
    ck.record(f"B4_kappa_{r['method'].split('>=')[0]}", None if r["kappa"] is None or pd.isna(r["kappa"]) else float(r["kappa"]))
total_requests = REQUESTS["chat"] + REQUESTS["embed"]
print("Requests sent in this session (cached replies not counted):", REQUESTS)
ck.record("total_requests_this_session", total_requests)
ck.check("B4_request_budget", total_requests <= 450, points=1,
         hint="Keep the lab under about 450 requests in one session. Re-running unchanged cells uses the cache; don't delete lab04_cache.json.")

# Written analysis

Answer in your own words, using **your numbers** from this notebook. Cite the chapter by section name where asked. A few sentences each is enough.

**Q1. When does each metric fail?** From your `scores` table, give one example (by qid) where exact match or token F1 got it wrong, and one where embedding similarity was misleading (hint: A3 and the "WRONG number" case). Explain why, using the chapter's discussion in *Exact Evaluation* (similarity measurements against reference data).

*Your answer:*

**Q2. Judge bias.** Report your self-preference index and each judge's first-position win rate and order consistency, plus what your A4 `verbose_correct` test showed. Do your numbers show self-bias, position bias, or verbosity bias? Connect them to the biases described in *Limitations of AI as a Judge*. Is 12 questions enough to be confident? (Hint: how many questions would have to change to flip your conclusion?) Name one change that would reduce the bias you found.

*Your answer:*

**Q3. Agreement with you, and what Larkspur should use.** Which method agreed best with your 10 labels (agreement and kappa), and where did a judge disagree with you? Use the chapter's points about *criteria ambiguity* and *inconsistency* to explain why a judge's score only means something together with its exact model and prompt. Then propose how Larkspur should evaluate the help-desk assistant every time the prompt or model changes: which metric(s), which judge, about how many questions, and the rough request cost per run (use this lab's request counts).

*Your answer:*

**Optional (not graded):** the chapter's *What Models Can Act as Judges?* section discusses using a model to judge itself (self-evaluation): it sounds like cheating because of self-bias, but it can be a useful sanity check. Would you let `qwen-moe` judge `qwen-moe` at Larkspur, and for what?

# AI-use reflection

This is the one reflection for all of Lab 04: include the Colab notebook too.

1. What did you use AI assistance for in this lab (tools, which steps, in both notebooks)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. You were asked to label 10 answers yourself. If you had asked an LLM to label them instead, what would your "agreement with a human" numbers have actually measured?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb.

In [ ]:
ck.report()